# YouTube Trending Videos — Bronze Layer Ingestion
### Raw CSV & JSON to Delta Lake (Medallion Architecture)

**Purpose:** This notebook ingests the raw YouTube trending videos dataset (CSV) and category metadata (JSON) from ADLS into the **bronze layer**. Each source is read as-is, enriched with audit columns, and written as a Delta table for downstream cleansing in the silver layer.

**Data Sources:**

| File | Format | Description |
|------|--------|-------------|
| `USvideos.csv` | CSV | ∼40K rows of US YouTube trending video stats (views, likes, comments, etc.) |
| `US_category_id.json` | JSON | Category ID → name mapping (nested `items[]` array) |

**Audit Columns Added:**
- `_bronze_ingested_at` — processing timestamp
- `_bronze_source_file` — full ADLS path of the source file
- `_bronze_batch_id` — unique UUID for batch traceability
- `_bronze_is_valid` — basic non-null check on the primary key column

---
_Downstream: the silver notebook cleanses these into a star schema (dim_date, dim_category, dim_channel, dim_video, fact_trending)._

In [0]:
# ── PySpark & Delta imports ──
# functions (aliased as f) → column-level operations (timestamps, lit, expr)
# DeltaTable → kept in scope for potential merge/upsert patterns later

from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# All YouTube data lives alongside the employee data in the same storage account,
# but under separate Unity Catalog schemas (bronze_youtube, silver_youtube, gold_youtube).

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema names ──
bronze_sch = "bronze_youtube"
silver_sch = "silver_youtube"
gold_sch = "gold_youtube"
youtube_db = "employeedatacatalog"

# Create all three schemas if they don't already exist
schema_data = [bronze_sch, silver_sch, gold_sch]

for schema in schema_data:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {youtube_db}.{schema}")

In [0]:
# ── Source file locations in ADLS ──
# USvideos.csv      → ~40K rows of trending video statistics
# US_category_id.json → nested JSON mapping category IDs to human-readable names

csv_source_data = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/source_data/USvideos.csv"
json_source_data = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/source_data/US_category_id.json"

In [0]:
# ============================================================
# STEP 1: Read both source files with schema inference
# ============================================================
# CSV options:
#   header=true   → first row contains column names
#   multiline=true → some fields (description, tags) span multiple lines
#   escape='"'     → handle double-quoted fields properly
#   inferSchema    → auto-detect column types (int, string, etc.)

df_raw = spark.read \
    .option("header", "true") \
    .option("multiline", "true") \
    .option("escape", '"') \
    .option("inferSchema", "true") \
    .csv(csv_source_data)

# JSON is read with multiline=true because the category file is a single JSON object
df_category_json = spark.read \
    .option("header", "true") \
    .option("multiline", "true") \
    .option("inferSchema", "true") \
    .json(json_source_data)

In [0]:
# ============================================================
# STEP 2: Add bronze-layer audit/metadata columns
# ============================================================
# These four columns enable downstream traceability and data quality:
#   _bronze_ingested_at  → when the row was processed
#   _bronze_source_file  → full ADLS path of the source file
#   _bronze_batch_id     → UUID for batch-level tracking
#   _bronze_is_valid     → basic non-null check on the primary key

# Video data — validity check on video_id (the natural key)
df_bronze = df_raw \
    .withColumn("_bronze_ingested_at", f.current_timestamp()) \
    .withColumn("_bronze_source_file", f.lit(csv_source_data)) \
    .withColumn("_bronze_batch_id", f.lit(f.expr("uuid()"))) \
    .withColumn("_bronze_is_valid", f.col("video_id").isNotNull())

# Category JSON — validity check on etag (present in every valid response)
df_bronze_category = df_category_json \
    .withColumn("_bronze_ingested_at", f.current_timestamp()) \
    .withColumn("_bronze_source_file", f.lit(json_source_data)) \
    .withColumn("_bronze_batch_id", f.lit(f.expr("uuid()"))) \
    .withColumn("_bronze_is_valid", f.col("etag").isNotNull())

In [0]:
# ============================================================
# STEP 3: Write both DataFrames as Delta tables to ADLS
# ============================================================
# Overwrite mode ensures a clean refresh on each run.
# Data lands under /bronze/videos and /bronze/category.

df_1 = df_bronze.write.format("delta").mode("overwrite").save(f"{bronze_path}/videos")
df_2 = df_bronze_category.write.format("delta").mode("overwrite").save(f"{bronze_path}/category")

In [0]:
# ============================================================
# STEP 4: Register Delta tables in Unity Catalog
# ============================================================
# This makes the bronze data queryable via SQL:
#   employeedatacatalog.bronze_youtube.raw_videos
#   employeedatacatalog.bronze_youtube.raw_category

df_1_table = spark.sql(f""" CREATE TABLE IF NOT EXISTS {youtube_db}.{bronze_sch}.raw_videos 
          using DELTA
          LOCATION '{bronze_path}/videos'""")

df_2_table = spark.sql(f""" CREATE TABLE IF NOT EXISTS {youtube_db}.{bronze_sch}.raw_category 
          using DELTA
          LOCATION '{bronze_path}/category'""")